# Track B — Fase 1: Ekstraksi Semua Backbone

**BDC Satria Data 2026** | 5 foundation model beku → cache embedding di Drive

| | |
|---|---|
| Backbone | SigLIP2-base, SigLIP2-so400m, SigLIP1-base, DINOv3-ViT-L, DINOv3-ConvNeXt-B |
| Varian | non-TTA + TTA (hflip+vflip, dirata-rata di level embedding) |
| Split | train (26.527) + test (1.458) |
| Output | `emb_*.npy` + manifest `.json` di Drive |

> **PRASYARAT — DINOv3 gated.** Sudah dikonfirmasi: kedua checkpoint DINOv3 menolak akses
> tanpa autentikasi. Sebelum menjalankan notebook ini:
> 1. Login HF, **terima lisensi** di `facebook/dinov3-vitl16-pretrain-lvd1689m`
>    dan `facebook/dinov3-convnext-base-pretrain-lvd1689m`
> 2. Buat token di `huggingface.co/settings/tokens`
> 3. Simpan sebagai **Colab Secret** bernama `HF_TOKEN` (ikon 🔑 di sidebar kiri),
>    aktifkan "Notebook access". **JANGAN** paste token langsung di cell — repo ini publik.

> **Copy gambar sudah otomatis di dalam script** (`local_cache.py`, copy paralel 32 thread).
> Tidak perlu cell copy manual, dan **jangan** set `FOLDS_CSV_OVERRIDE`.

---
## 🔧 SETUP

In [ ]:
# Cell 1 — Clone / update repo
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull
print('✅ Repo siap')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers scikit-learn lightgbm
# hf_xet (backend download baru HF Hub) 401 pada request anonim untuk sebagian
# file. Uninstall memaksa fallback HTTP biasa, yang jalan andal.
!pip uninstall -y hf_xet hf-xet
print('✅ dependencies siap (hf_xet dilepas)')

In [ ]:
# Cell 3 — HF token (DINOv3 gated). Diambil dari Colab Secrets, BUKAN di-hardcode.
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('✅ HF_TOKEN ter-set' if os.environ.get('HF_TOKEN') else '❌ HF_TOKEN KOSONG')

In [ ]:
# Cell 4 — Mount Drive + verifikasi GPU
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ TIDAK ADA GPU')

import os
for p in [
    '/content/drive/MyDrive/BDC2026apace/output_trackA/folds.csv',
    '/content/drive/MyDrive/BDC2026apace/submission.csv',
    '/content/drive/MyDrive/BDC2026apace/test',
    '/content/drive/MyDrive/BDC2026apace/output_trackB/embeddings',
]:
    print(('✅' if os.path.exists(p) else '❌ TIDAK ADA'), p)

---
## 🚀 EKSTRAKSI

Urutan yang akan terjadi:
1. **Preflight** (~5 mnt) — load kelima backbone, cek masing-masing menghasilkan embedding.
   Kalau DINOv3 gated/token salah, **gagal di sini**, bukan setelah 2 jam ekstraksi.
2. **Copy gambar ke disk lokal** — paralel, sekali saja, resume-safe.
3. **Ekstraksi** per backbone (model di-load sekali, VRAM dibersihkan antar backbone).
   `siglip2b256` non-TTA otomatis di-skip (sudah ada dari safety net).

In [ ]:
# Cell 5 — Jalankan extract_all.py (GPU, berjam-jam)
import os
os.chdir('/content/satria-data-bdcugm02/track_b/src')
!python ../experiments/extract_all.py

In [ ]:
# Cell 6 -- Verifikasi semua embedding ada di Drive
import os
import sys
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
from config import CFG

NAMES = ['siglip2b256', 'siglip2so400m', 'siglip1b256', 'dinov3vitl', 'dinov3cnxb']
missing = []
for name in NAMES:
    for suffix in ['', '_tta']:
        for split in ['train', 'test']:
            fname = f'{name}{suffix}_{split}.npy'
            ok = os.path.exists(os.path.join(CFG.embeddings_dir, fname))
            if not ok:
                missing.append(fname)
            print('OK  ' if ok else 'HILANG', fname)

if missing:
    print(len(missing), 'file belum ada -- jalankan ulang cell ekstraksi (resume otomatis)')
else:
    print('LENGKAP -- lanjut ke probe_grid.py')
